# Pareto-Dominance DPO v2: Full 0-9 Rollout Pool (Kaggle 2xT4)

Retrains Phase 3 (Pareto-dominance-filtered DPO) on `experiments/018_pareto_dpo_v2_extra_rollouts/data/pareto_dpo_pairs_v2.jsonl` (298 pairs from 117/309 questions), built by combining the original 0-4 rollouts with the newly-judged 5-9 rollouts (experiment 016) -- v1 only had 154 pairs from 73/309 questions using rollouts 0-4 alone. Same Pareto-dominance filtering logic, same trainer, same hyperparameters as v1 -- the one variable being tested here is candidate-rollout-pool size, not pair-selection method (already tested in v1) or judge (already tested in Phase 2).


In [ ]:
# Cell 1: Check GPU hardware and install sm_60 compatible PyTorch stack if Tesla P100 is assigned
import os, subprocess, sys, torch

print(f'Initial PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {props.name}, Compute Capability: {cc}, VRAM: {props.total_memory / 1e9:.1f} GB')
    if cc[0] < 7:
        print(f'*** Tesla P100 (cc {cc}) detected. PyTorch 2.12 dropped sm_60 CUDA kernels.')
        print('*** Installing PyTorch 2.5.1+cu124 with full sm_60 CUDA GPU support...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
packages = ['transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU device: {torch.cuda.get_device_name(0)}')
print('Packages successfully configured.')


In [ ]:
# Cell 2: Checkout repository and execute Pareto-dominance DPO v2 trainer
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
# main has all Phase 1-3 DG-PRM code and data -- plain clone defaults to main.
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

pairs_path = repo_dir / 'experiments/018_pareto_dpo_v2_extra_rollouts/data/pareto_dpo_pairs_v2.jsonl'
assert pairs_path.exists(), f'Missing {pairs_path} -- commit/push problem.'
with open(pairs_path, encoding='utf-8') as f:
    n = sum(1 for line in f if line.strip())
assert n == 298, f'Expected 298 pairs, found {n}'
print(f'Confirmed {n} Pareto DPO v2 pairs present after checkout.')

# Run Pareto-dominance DPO training -- same trainer, same hyperparameters as v1 and the
# existing Full DPO kernel, only --dataset-path/--output-dir differ.
env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/train/train_dpo.py',
    '--dataset-path', 'experiments/018_pareto_dpo_v2_extra_rollouts/data/pareto_dpo_pairs_v2.jsonl',
    '--output-dir', '/kaggle/working/qwen_vl_pareto_dpo_v2_adapter',
    '--epochs', '1',
    '--batch-size', '1',
    '--lr', '1e-5',
    '--beta', '0.1'
]
subprocess.run(cmd, env=env, check=True)


In [ ]:
# Cell 3: Validate output artifacts
out_dir = Path('/kaggle/working/qwen_vl_pareto_dpo_v2_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'Adapter directory {out_dir} contents: {files}')
assert (out_dir / 'adapter_config.json').exists(), 'adapter_config.json missing -- training did not save a LoRA adapter.'
